# Test Visualization for SegFormer-B2 on Crack500

This notebook loads the saved best `SegFormer-B2` checkpoint and visualizes predictions on the `Crack500` test set.

Use it to:
- inspect qualitative prediction quality on fixed test samples
- compare ground-truth masks and predicted masks side by side
- compare the same samples against the earlier U-Net notebook


In [ ]:
import os
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from dataset import CrackDataset
from loss import BCEDiceLoss
from metrics import f1_score, iou_score
from model import get_model

ROOT = "../CRACK500"
img_size = 360
batch_size = 8
threshold = 0.5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = os.path.join("..", "checkpoints", "segformer_b2_best.pth")

model_name = "SegFormer-B2"
encoder_name = "resnet34"
encoder_weights = "imagenet"
in_channels = 3
classes = 1

# Keep the same fixed samples as the U-Net notebook for direct visual comparison.
sample_indices = [0, 5, 12, 25, 40]

plt.rcParams["figure.dpi"] = 120

print(f"Using device: {device}")
print(f"Checkpoint path: {checkpoint_path}")
print(f"Fixed sample indices: {sample_indices}")


## 1. Load the Test Dataset

We keep the dataset setup identical to the training and test scripts so the notebook stays consistent with the recorded baseline.

In [ ]:
test_dataset = CrackDataset(ROOT, split="test", img_size=img_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Loaded test dataset with {len(test_dataset)} samples.")
raw_image, raw_mask = test_dataset.get_raw(0)
print(f"Sample 0 raw image shape: {raw_image.shape}")
print(f"Sample 0 raw mask shape: {raw_mask.shape}")


## 2. Rebuild the Model and Load the Checkpoint

The checkpoint stores learned weights, not the model definition itself, so we recreate the same model and then load the saved `state_dict`.

In [ ]:
model = get_model(
    model_name=model_name,
    encoder_name=encoder_name,
    encoder_weights=encoder_weights,
    in_channels=in_channels,
    classes=classes,
)
model = model.to(device)
criterion = BCEDiceLoss()

if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(
        f"Checkpoint not found at: {checkpoint_path}. "
        "Make sure training has saved segformer_b2_best.pth before running this notebook."
    )

state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model.eval()

print("Model rebuilt and checkpoint loaded successfully.")


## 3. Define Helper Functions

We use the transformed sample for inference, then resize the predicted mask back to the raw image size for clean plotting.

In [ ]:
def make_overlay(image_np, mask_np, color=(255, 0, 0), alpha=0.45):
    overlay = image_np.copy().astype(np.float32)
    color_arr = np.array(color, dtype=np.float32)
    overlay[mask_np == 1] = overlay[mask_np == 1] * (1 - alpha) + color_arr * alpha
    return overlay.clip(0, 255).astype(np.uint8)


def predict_one(index, threshold=threshold):
    raw_image, raw_mask = test_dataset.get_raw(index)

    image_t, mask_t = test_dataset[index]
    image_t = image_t.unsqueeze(0).to(device)
    mask_t = mask_t.unsqueeze(0).unsqueeze(1).float().to(device)

    with torch.no_grad():
        logits = model(image_t)
        probs = torch.sigmoid(logits)
        pred_mask_t = (probs > threshold).float()
        loss = criterion(logits, mask_t).item()
        iou = iou_score(logits, mask_t, threshold=threshold)
        f1 = f1_score(logits, mask_t, threshold=threshold)

    raw_h, raw_w = raw_mask.shape
    pred_mask = F.interpolate(
        pred_mask_t,
        size=(raw_h, raw_w),
        mode="nearest",
    )[0, 0].cpu().numpy().astype(np.uint8)

    pred_overlay = make_overlay(raw_image, pred_mask)
    gt_overlay = make_overlay(raw_image, raw_mask)

    return {
        "image": raw_image,
        "gt_mask": raw_mask,
        "pred_mask": pred_mask,
        "gt_overlay": gt_overlay,
        "pred_overlay": pred_overlay,
        "loss": loss,
        "iou": iou,
        "f1": f1,
    }


## 4. Visualize Fixed Test Samples

These are the same fixed indices used in the U-Net visualization notebook, so you can compare them one by one.

In [ ]:
num_rows = len(sample_indices)
fig, axes = plt.subplots(num_rows, 5, figsize=(16, 3 * num_rows))

if num_rows == 1:
    axes = np.expand_dims(axes, axis=0)

column_titles = ["Image", "Ground Truth", "GT Overlay", "Prediction", "Prediction Overlay"]
for col, title in enumerate(column_titles):
    axes[0, col].set_title(title, fontsize=11)

for row, idx in enumerate(sample_indices):
    sample = predict_one(idx)

    axes[row, 0].imshow(sample["image"])
    axes[row, 1].imshow(sample["gt_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 2].imshow(sample["gt_overlay"])
    axes[row, 3].imshow(sample["pred_mask"], cmap="gray", vmin=0, vmax=1)
    axes[row, 4].imshow(sample["pred_overlay"])

    axes[row, 0].set_ylabel(
        f"idx={idx}\nloss={sample['loss']:.3f}\nIoU={sample['iou']:.3f}\nF1={sample['f1']:.3f}",
        rotation=0,
        labelpad=45,
        va="center",
        fontsize=9,
    )

    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()


## 5. Sanity Check the Whole Test Set

This cell recomputes full-test metrics inside the notebook so you can quickly verify that the loaded checkpoint matches the CLI evaluation script.

In [ ]:
test_loss = 0.0
test_iou = 0.0
test_f1 = 0.0

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.unsqueeze(1).float().to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        batch_size_now = images.size(0)
        test_loss += loss.item() * batch_size_now
        test_iou += iou_score(outputs, masks, threshold=threshold) * batch_size_now
        test_f1 += f1_score(outputs, masks, threshold=threshold) * batch_size_now

test_loss /= len(test_dataset)
test_iou /= len(test_dataset)
test_f1 /= len(test_dataset)

print(f"Notebook test check | loss={test_loss:.4f} | iou={test_iou:.4f} | f1={test_f1:.4f}")
print()
print("Fixed-sample summary:")

for idx in sample_indices:
    sample = predict_one(idx)
    gt_pixels = int(sample["gt_mask"].sum())
    pred_pixels = int(sample["pred_mask"].sum())
    print(
        f"idx={idx:4d} | loss={sample['loss']:.4f} | "
        f"IoU={sample['iou']:.4f} | F1={sample['f1']:.4f} | "
        f"gt_pixels={gt_pixels} | pred_pixels={pred_pixels}"
    )
